In [1]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd


print("Sedang memuat Model Semantik (SBERT)... (Hanya memakan waktu di awal)")
model_sbert = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("Model SBERT siap digunakan!")

c:\Users\USER\Desktop\sistem_rekomendasi\server\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Sedang memuat Model Semantik (SBERT)... (Hanya memakan waktu di awal)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4183.31it/s]


Model SBERT siap digunakan!


In [2]:
import re

def hitung_rekomendasi(data_dosen, judul_mhs, abstrak_mhs, k_rank, bobot_lexical, bobot_semantic):
    teks_mahasiswa = judul_mhs + " " + abstrak_mhs
    
    # 1. Siapkan Corpus Dosen
    teks_semua_dosen = []
    teks_gabungan_raw = [] 
    
    for dosen in data_dosen:
        teks_gabungan = f"{dosen.get('RIWAYAT_PENDIDIKAN','')} {dosen.get('BIDANG_KEAHLIAN','')} " \
                        f"{dosen.get('JURNAL','')} {dosen.get('judul uji','')} {dosen.get('judul bimbing','')}"
        teks_gabungan_raw.append(teks_gabungan)
        teks_semua_dosen.append(teks_gabungan.lower())
        
    # 2. PROSES SKORING GLOBAL (BM25 & SBERT)
    token_dosen = [teks.split() for teks in teks_semua_dosen]
    token_mahasiswa_list = teks_mahasiswa.lower().split()
    
    mesin_bm25 = BM25Okapi(token_dosen)
    skor_bm25_mentah = mesin_bm25.get_scores(token_mahasiswa_list)
    nilai_tertinggi_bm25 = max(skor_bm25_mentah) if max(skor_bm25_mentah) > 0 else 1
    skor_leksikal = [skor / nilai_tertinggi_bm25 for skor in skor_bm25_mentah]

    vektor_mahasiswa = model_sbert.encode([teks_mahasiswa])
    vektor_dosen = model_sbert.encode(teks_semua_dosen)
    skor_semantik = cosine_similarity(vektor_mahasiswa, vektor_dosen)[0]

    # 3. GABUNGKAN & URUTKAN SKOR HYBRID
    semua_hasil = []
    for indeks in range(len(data_dosen)):
        skor_hybrid = (bobot_lexical * skor_leksikal[indeks]) + (bobot_semantic * skor_semantik[indeks])
        semua_hasil.append({
            "indeks_asli": indeks,
            "NAMA": data_dosen[indeks]['NAMA'],
            "Hybrid Score": skor_hybrid,
            "Lexical Score": skor_leksikal[indeks],
            "Semantic Score": skor_semantik[indeks]
        })
        
    # Urutkan dan ambil Top K
    semua_hasil.sort(key=lambda x: x["Hybrid Score"], reverse=True)
    top_k_hasil = semua_hasil[:k_rank]
    
    # ==========================================
    # PROSES 4: VISUALISASI KATA KUNCI (HANYA UNTUK TOP K)
    # ==========================================
    # Daftar kata sambung bahasa Indonesia sederhana untuk diabaikan
    stopwords = {'yang', 'untuk', 'pada', 'dari', 'dengan', 'dalam', 'dan', 'atau', 'ini', 'itu', 'sebagai', 'oleh', 'ke', 'di', 'berbasis', 'sistem', 'menggunakan'}
    
    # Bersihkan teks mahasiswa dari simbol untuk pencarian leksikal
    teks_mhs_bersih = set(re.findall(r'\b[a-z]{3,}\b', teks_mahasiswa.lower())) - stopwords

    hasil_akhir = []
    for hasil in top_k_hasil:
        idx = hasil["indeks_asli"]
        teks_dosen_terpilih = teks_semua_dosen[idx]
        
        # A. KATA KUNCI LEKSIKAL (Kata yang sama persis)
        teks_dsn_bersih = set(re.findall(r'\b[a-z]{3,}\b', teks_dosen_terpilih)) - stopwords
        kata_leksikal = list(teks_mhs_bersih.intersection(teks_dsn_bersih))
        
        # B. KATA KUNCI SEMANTIK (Kata dosen yang maknanya paling relevan dengan mahasiswa)
        kata_semantik = []
        kandidat_kata_dosen = list(teks_dsn_bersih)
        
        if kandidat_kata_dosen:
            # Ubah kata-kata unik dosen menjadi vektor, bandingkan dengan vektor mahasiswa utuh
            vektor_kandidat = model_sbert.encode(kandidat_kata_dosen)
            skor_kemiripan_kata = cosine_similarity(vektor_mahasiswa, vektor_kandidat)[0]
            
            # Ambil 4 kata teratas yang paling kuat ikatan semantiknya
            indeks_kata_teratas = skor_kemiripan_kata.argsort()[-4:][::-1]
            kata_semantik = [kandidat_kata_dosen[i] for i in indeks_kata_teratas]

        # Susun data final
        hasil_akhir.append({
            "NAMA DOSEN": hasil['NAMA'],
            "Hybrid Score": round(float(hasil['Hybrid Score']), 3),
            "Irisan Kata (Lexical)": ", ".join(kata_leksikal) if kata_leksikal else "-",
            "Makna Terkait (Semantic)": ", ".join(kata_semantik) if kata_semantik else "-"
        })
        
    return hasil_akhir

In [ ]:
# Dummy Data Dosen untuk testing di Notebook
judul_input = "Implementasi Deep Learning untuk Klasifikasi Gambar"
abstrak_input = "Penelitian ini menggunakan algoritma berbasis kecerdasan buatan untuk mendeteksi objek."

DATASET_PATH = "C:\Users\USER\Desktop\sistem_rekomendasi\server\data\dataset_profiles_terintegrasi.xlsx"

dataset_dosen = pd.read_excel(DATASET_PATH)

hasil = hitung_rekomendasi(
    data_dosen=dataset_dosen.to_dict('records'),  
    judul_mhs=judul_input,
    abstrak_mhs=abstrak_input,
    k_rank=5,                 
    bobot_lexical=0.4,      
    bobot_semantic=0.6
)

# 2. Tampilkan DataFrame
df_hasil = pd.DataFrame(hasil)

# Mengatur tampilan Pandas agar teks tidak terpotong (opsional tapi disarankan)
pd.set_option('display.max_colwidth', None)

# Tampilkan
df_hasil

SyntaxError: (unicode error) 'unicodeescape' codec can't decode bytes in position 2-3: truncated \UXXXXXXXX escape (1040790672.py, line 5)